In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import json
from pathlib import Path
import pandas as pd
import requests
from ift6758.llm.get_better import (
    extract_text,
    load_function,
    format_test_report,
    build_refine_prompt,
    all_passed,
)
from ift6758.llm.chat import confirm, ask_user
from ift6758.llm.prompts import PROFILES, OG_TACHE_1
from ift6758.llm.qwen_log import create_log, log_call
from ift6758.llm.qwen_utils import (
    load_qwen,
    build_messages,
    generate,
    ask_qwen,
    check_env,
)
from ift6758.llm.rag import (
    load_doc,
    extract_section,
    split_doc,
    load_embedder,
    embed,
    retrieve,
    format_context,
    DOCS_DIR,
)
from ift6758.llm.test_both_code import run_tests

check_env()  # Demo 2, for reproductibility

Python: 3.11.15
NumPy: 1.26.4 | pandas: 2.3.3
Transformers: 5.17.0
Sentence Transformers: 6.1.0
CUDA GPU: Unavailable — CPU/Apple GPU fallback


In [6]:
zmalski_file = "zmalski-nhl-api-reference.md"
zmalski_url = (
    "https://raw.githubusercontent.com/Zmalski/NHL-API-Reference/main/README.md"
)

text = load_doc(zmalski_url, zmalski_file)
section = extract_section(
    text, "# NHL Web API Documentation", "# NHL Stats API Documentation"
)
chunks = split_doc(section)
embedder = load_embedder()
vectors = embed(chunks, embedder)
p_gen = PROFILES["gen_code"]
results = retrieve(p_gen["retrieval_query"], chunks, vectors, embedder, k=p_gen["k"])
context = format_context(results, source=zmalski_file)
print(context)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[1] (source:zmalski-nhl-api-reference.md, passage 35) ###### Example using cURL:

```bash
curl -X GET "https://api-web.nhle.com/v1/where-to-watch"
```

### Game Events
#### Get Play By Play
- **Endpoint**: `/v1/gamecenter/{game-id}/play-by-play`
- **Method**: GET
- **Description**: Retrieve play-by-play information for a specific game.
- **Parameters**:
  - `game-id` (int) - Game ID
- **Response**: JSON format

###### Example using cURL:

```bash
curl -X GET "https://api-web.nhle.com/v1/gamecenter/2023020204/play-by-play"
```

[2] (source:zmalski-nhl-api-reference.md, passage 60) ###### Example using cURL:

```bash
curl -X GET "https://api-web.nhle.com/v1/gamecenter/2023020204/right-rail"
```

#### Get WSC Play By Play
- **Endpoint**: `/v1/wsc/play-by-play/{game-id}`
- **Method**: GET
- **Description**: Retrieves WSC (World Showcase) play-by-play information for a specific game.
- **Parameters**:
  - `game-id` (int) - Game ID
- **Response**: JSON format

###### Example using cURL:

```

In [4]:
# Code adapté de la démo 2
GAME_ID = 2022030411
raw_path = Path(DOCS_DIR.parent / "raw") / f"{GAME_ID}.json"
print("Cached game:", raw_path.resolve())
if not raw_path.exists():
    response = requests.get(
        f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play", timeout=30
    )
    response.raise_for_status()
    downloaded = response.json()
    assert downloaded["id"] == GAME_ID and isinstance(downloaded["plays"], list)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(response.text, encoding="utf-8")
game = json.loads(
    raw_path.read_text(encoding="utf-8")
)  # Convertie de json vers objet python
assert game["id"] == GAME_ID and isinstance(game["plays"], list)
print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

Cached game: /Users/jcforget/Maitrise/Cours/IFT 6758/Projet/Github/NHL-data-analysis/ift6758/data/raw/2022030411.json
FLA at VGK 2023-06-03


In [5]:
model, tokenizer, device = load_qwen()
log_path = create_log()
print("Log :", log_path)


# Code suggested by Claude to embelish presentation
def banner(title):
    print(f"\n{'=' * 15} {title} {'=' * 15}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Log : /Users/jcforget/Maitrise/Cours/IFT 6758/Projet/Github/NHL-data-analysis/logs/chat_2026-10-15_20-47-48.jsonl


In [8]:
MAX_ITER = 5  # SafeGuard
task = OG_TACHE_1
task_version = "game_v0"  # first version (demo 2 ift6758)
iteration = 0
history = []  # Loop resume, seen at the end

try:
    while True:
        # Code genration section
        banner(f"Step {iteration} : code generation ({task_version})")
        print(task)
        gen_result = ask_qwen(
            model=model,
            tokenizer=tokenizer,
            log_path=log_path,
            prompt=task,
            context=context,
            profile=PROFILES["gen_code"],
            extra={
                "iteration": iteration,
                "task_version": task_version,
                "retrieved": results,
            },
        )

        # Extraction and humain in the lopp validation
        try:
            game_code = extract_text(gen_result["answer"], tag="code")
        except ValueError as e:
            game_code = gen_result["answer"]  # no code generatde
            test_results = {"0. Extraction code": f"ERROR : {e}"}
        else:
            banner("Extracted code")
            print(game_code)
            if not confirm("Execute and test code ?"):
                print("Stawp : code ist nicht gut.")
                break
            # Transforme
            try:
                fn = load_function(game_code, "fetch_game_llm")
            except Exception as e:  # SyntaxError, import manquant...
                test_results = {"0. Code loading": f"ERROR : {type(e).__name__}: {e}"}
            else:
                test_results = run_tests(fn, GAME_ID, game)
        report = format_test_report(test_results)
        banner("Résultats des tests")
        print(report)
        history.append(
            {
                "iteration": iteration,
                "task_version": task_version,
                "passed": all_passed(test_results),
                "report": report,
            }
        )
        if all_passed(test_results):
            print("\nAll test are Ok !")
            break
        if iteration >= MAX_ITER:
            print(f"\n MAX_ITER ({MAX_ITER}) too many iteration")
            break
        # Prompt refinement
        banner(f"Step {iteration} : prompt refinement")
        refine_prompt = build_refine_prompt(
            task=task,
            code=game_code,
            context=context,
            report=report
        )  # Raffine le prompt
        refine_result = ask_qwen(
            model=model,
            tokenizer=tokenizer,
            log_path=log_path,
            prompt=refine_prompt,
            context=None,
            profile=PROFILES["refine"],
            extra={
                "iteration": iteration,
                "task_version": task_version,
                "test_results": test_results,
            },
        )
        try:
            new_task = extract_text(
                answer=refine_result["answer"], tag="new_prompt"
            ).strip()
        except ValueError as e:
            print(f"Stawp!!! : {e}")
            break
        banner("New prompt")
        print(new_task)
        if not confirm("Accept new prompt and generate new code ?"):
            print("Stawp : prompt ist nicht gut.")
            break
        # Netx task
        task = new_task
        iteration += 1
        task_version = f"game_v{iteration}"
except KeyboardInterrupt:
    print(f"\n Stopped at step {iteration}.")
pd.DataFrame(history)


=============== Step 0 : code generation (og1) ===============

Write a Python function fetch_game_llm(game_id, cache_dir).
Requirements from our project:
- Use requests, pathlib and json. Include imports and a docstring.
- The API host is https://api-web.nhle.com.
- If cache_dir/<game_id>.json exists, load it with no network request.
- Otherwise GET play-by-play with timeout=30 and raise_for_status().
- Our observed response has id (int) and plays (list). Validate both.
- Create the cache directory and save valid JSON as UTF-8.
- Return the full game dictionary. Validate cached data too.
Return only the function and imports in a Python code block.

```python
import requests
import json
from pathlib import Path

def fetch_game_llm(game_id, cache_dir):
    """
    Fetch play-by-play data for a given game ID from the NHL API with caching.
    
    Args:
        game_id (int): The ID of the game to fetch data for.
        cache_dir (str): Directory path where cached JSON files will be st

,iteration,task_version,passed,report
0,0,og1,False,"3/4 tests passed\nTest: 1. Known game, Status:..."
1,1,revision-1,True,"4/4 tests passed\nTest: 1. Known game, Status:..."


In [ ]:
logs = pd.read_json(log_path, lines=True)
logs[
    ["iteration", "profile", "task_version", "n_token_in", "n_token_out", "duration_s"]
]

Ici, on va réutilisé le code pour une game plutôt qu'essayer de tout refaire, ce serait vraiment trop long.

In [ ]:
# Prompt basé sur la démo 2 avec des informations venant de dword4. Modifié pour fetch season au lieu de game. Structure proposé par Qwen et par Claude
game_block = f"""
In a previous step, you wrote this function:
```python
{game_code}
```
Do not rewrite it. Call it.
"""
season_task = """
Write a Python function fetch_season_llm(season, game_type, cache_dir).
Goal: fetch, cache and load all NHL play-by-play games of one season and one game type, by calling fetch_game_llm(game_id, cache_dir) for each game.

Game ID format (10 digits), from the NHL API documentation:
- will look like this : 2023020001.
- first 4 digits: season start year (2016 for the 2016-2017 season).
- next 2 digits: game type, 02 = regular season, 03 = playoffs.
- last 4 digits: game number
     - regular season: 0001, 0002, ... until the first game that does not exist.
     - playoffs: 0 + round (1 to 4) + matchup + game (1 to 7). Rounds 1, 2, 3, 4 have 8, 4, 2, 1 matchups.

Requirements:
- season is an int (e.g. 2016). game_type is 2 or 3.
- Build each game_id as an int.
- A game does not exist when fetch_game_llm raises requests.HTTPError with response.status_code == 404. Regular season: stop at the first missing game. Playoffs: skip to the next matchup.
- Do not catch any other error: let it propagate.
- Return a list of game dicts, sorted by game_id.
- Include imports and a docstring.
Return only fetch_season_llm and its imports in a Python code block.
"""

In [ ]:
MAX_ITER = 5  # SafeGuard
task = season_task
task_version = "season_v0"  # first version (veiw cell)
season_iteration = 0
season_history = []  # Loop resume, seen at the end

try:
    while True:
        # Code genration section
        banner(f"Step {season_iteration} : code generation ({task_version})")
        print(task)
        gen_result = ask_qwen(
            model=model,
            tokenizer=tokenizer,
            log_path=log_path,
            prompt=game_block + "\n\n" + task,
            context=None,  # À voir si on veut utiliser l'API ou pas, en particulier celle de dword4
            profile=PROFILES["gen_code"],
            extra={
                "iteration": season_iteration,
                "task_version": task_version,
                # "retrieved": results,
            },
        )

        # Extraction and humain in the lopp validation
        try:
            season_code = extract_text(gen_result["answer"], tag="code")
        except ValueError as e:
            season_code = gen_result["answer"]  # no code generatde
            test_results = {"0. Extraction code": f"ERROR : {e}"}
        else:
            banner("Extracted code")
            print(season_code)
            if not confirm("Execute and test code ?"):
                print("Stawp : code ist nicht gut.")
                break
            # Transforme
            try:
                fn = load_function(game_code + "\n\n" + season_code, "fetch_season_llm")
            except Exception as e:  # SyntaxError, import manquant...
                test_results = {"0. Code loading": f"ERROR : {type(e).__name__}: {e}"}
            else:
                test_results = run_tests(fn, GAME_ID, game)
        report = format_test_report(test_results)
        banner("Résultats des tests")
        print(report)
        season_history.append(
            {
                "iteration": season_iteration,
                "task_version": task_version,
                "passed": all_passed(test_results),
                "report": report,
            }
        )
        if all_passed(test_results):
            print("\nAll test are Ok !")
            break
        if season_iteration >= MAX_ITER:
            print(f"\n MAX_ITER ({MAX_ITER}) too many iteration")
            break
        # Prompt refinement
        banner(f"Step {season_iteration} : prompt refinement")
        refine_prompt = build_refine_prompt(
            task=task,
            code=season_code,
            context=None,
            report=report
        )  # Raffine le prompt
        refine_result = ask_qwen(
            model=model,
            tokenizer=tokenizer,
            log_path=log_path,
            prompt=refine_prompt,
            context=None,
            profile=PROFILES["refine"],
            extra={
                "iteration": season_iteration,
                "task_version": task_version,
                "test_results": test_results,
            },
        )
        try:
            new_task = extract_text(refine_result["answer"], tag="new_prompt").strip()
        except ValueError as e:
            print(f"Stawp!!! : {e}")
            break
        banner("New prompt")
        print(new_task)
        if not confirm("Accept new prompt and generate new code ?"):
            print("Stawp : prompt ist nicht gut.")
            break
        # Netx task
        task = new_task
        season_iteration += 1
        task_version = f"season_v{season_iteration}"
except KeyboardInterrupt:
    print(f"\n Stopped at step {season_iteration}.")
pd.DataFrame(season_history)